This note shows how to use SABRE to transform a QuantumCircuit. The objective is either depth or SWAP count.
https://qiskit.org/documentation/tutorials/circuits_advanced/04_transpiler_passes_and_passmanager.html
"This is a design philosophy of Qiskit’s transpiler: every pass performs a small, well-defined action, and the aggressive circuit optimization is achieved by the pass manager through combining multiple passes."

Note we don't use SabreLayout directly; instead, we restructured it as sabre_mapper so that the seed can be easilly changed every time. A question: is the SabreSwap used in sabre_mapper has attribute fakerun=True?

## Issues with SabreSwap
0. The seed in sabre_mapper seems to play an important role.
1. Running for example "excitation_preserving_6.qasm" on grid2x3 obtains much better depth result than on tokyo! (Checked again on Oct 9 but obtianed the contrary result!) 
2. Sabre is rather unstable on say "grover_operator_14".
3. SabreSwap is greatly affected by its initial mapping: for 'grover_operator_14', five runs may have results:
      Transformed circuit: swap count = 1321, final depth = 41189
      Transformed circuit: swap count = 2129, final depth = 41382
      Transformed circuit: swap count = 1066, final depth = 41113
      Transformed circuit: swap count = 951, final depth = 41863
      Transformed circuit: swap count = 848, final depth = 41190

In [3]:
import copy
import random
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler.layout import Layout
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.passes.layout.set_layout import SetLayout
from qiskit.transpiler.passes.layout.full_ancilla_allocation import FullAncillaAllocation
from qiskit.transpiler.passes.layout.enlarge_with_ancilla import EnlargeWithAncilla
from qiskit.transpiler.passes.layout.apply_layout import ApplyLayout
from qiskit.transpiler.passmanager import PassManager
from qiskit.circuit.quantumregister import Qubit, QuantumRegister

import matplotlib.pyplot as plt


#from qiskit.converters import dag_to_circuit, circuit_to_dag
#from qiskit.dagcircuit import DAGCircuit, DAGOpNode, DAGInNode, DAGOutNode

In [4]:
#The following four functions are part of functions of SabreLayout
def _layout_and_route_passmanager(initial_layout, coupling_map, test):
    if test:
        from sabre_swap_bridge import SabreSwap
    else:
        from qiskit.transpiler.passes.routing import SabreSwap
        #from sabre_swap0330 import SabreSwap
    layout_and_route = [
            SetLayout(initial_layout),
            FullAncillaAllocation(coupling_map),
            EnlargeWithAncilla(),
            ApplyLayout(),
            SabreSwap(coupling_map, heuristic = 'lookahead'),
        ]
    pm = PassManager(layout_and_route)
    return pm

def _compose_layouts(initial_layout, pass_final_layout, qregs):
    trivial_layout = Layout.generate_trivial_layout(*qregs)
    qubit_map = Layout.combine_into_edge_map(initial_layout, trivial_layout)
    final_layout = {v: pass_final_layout._v2p[qubit_map[v]] for v in initial_layout._v2p}
    return Layout(final_layout)

"""Transform the input circuit and generate an initial layout"""
def sabre_mapper(cir_in: QuantumCircuit, coupling_map: CouplingMap):
    circ = cir_in.copy()
    #circ.draw('mpl')
    """everytime use a different seed"""
    seed = np.random.randint(0, np.iinfo(np.int32).max) 
    rng = np.random.default_rng(seed)

    physical_qubits = rng.choice(coupling_map.size(), len(circ.qubits), replace=False)
    physical_qubits = rng.permutation(physical_qubits)
    initial_layout = Layout({q: circ.qubits[i] for i, q in enumerate(physical_qubits)})
    
    
    max_iterations = 3
    rev_circ = circ.reverse_ops()
    for i in range(max_iterations):
        for _ in ("forward", "backword"):
            pm = _layout_and_route_passmanager(initial_layout, coupling_map, test=False)
            new_circ = pm.run(circ)
            #new_circ.draw('mpl')
            pass_final_layout = pm.property_set["final_layout"]
            final_layout = _compose_layouts(initial_layout, pass_final_layout, new_circ.qregs)
            initial_layout = final_layout
            circ, rev_circ = rev_circ, circ
    return circ, initial_layout

"""One SABRE run with output = number of swaps"""            
def execute_sabre(circ: QuantumCircuit, coupling_map: CouplingMap):
    newcirc1, initial_layout1 = sabre_mapper(circ, coupling_map)
    pm_final_true = _layout_and_route_passmanager(initial_layout1, coupling_map, test=True)
    circ_final_true = pm_final_true.run(newcirc1)    

    #print(circ_final_true.count_ops())
    if 'swap' in  circ_final_true.count_ops():
        num_swap_true = circ_final_true.count_ops()['swap']
    else:
        num_swap_true = 0
    
    #newcirc2, initial_layout2 = sabre_mapper(circ, coupling_map)
    pm_final_false = _layout_and_route_passmanager(initial_layout1, coupling_map, test=False)
    circ_final_false = pm_final_false.run(newcirc1)    
    if 'swap' in  circ_final_false.count_ops():
        num_swap_false = circ_final_false.count_ops()['swap']
    else:
        num_swap_false = 0
        
    #circ_final = circ_final.decompose("swap")
    #depth_overhead = circ_final.depth()-circ.depth()
    #return num_swap, depth_overhead  
    return num_swap_true, num_swap_false

In [27]:
from ag import qgrid, q20, rochester, Sycamore54Q, guadalupe
import time
from qiskit.visualization import circuit_drawer
from dac_part import remove_1q_and_consecutive_2q_gates_in_circuit

#path = '../bench/MQTBench/'
#filename = 'ghz_indep_qiskit_53.qasm'

#path = '../bench/effectiveQM/'

path = '../bench/qiskit_circuit_benchmark/' 
filename = 'grover_operator_6.qasm'


#AG = guadalupe()
#AG = q20()
AG = qgrid(2,3)
#AG = Sycamore54Q()
#print(AG.edges())


coupling_map = CouplingMap(AG.edges())
repeat = 10
obj = 'swap_overhead'

"""Create a QuantumCircuit from the QASM file"""
with open(path+filename, 'r') as file:
    qasm_code = file.read()
qc = QuantumCircuit.from_qasm_str(qasm_code)
#qc = remove_1q_and_consecutive_2q_gates_in_circuit(circ)

#dag = circuit_to_dag(newcirc)
#print(filename, dag.count_ops(), dag.depth())

print(f"{filename}: qubit number={qc.num_qubits}, input depth={qc.depth()}, \n  ----- {qc.count_ops()}")

timeA = time.time()

for i in range(repeat):
    num_swap_true, num_swap_false = execute_sabre(qc, coupling_map)
    #if num_swap_true < 720 or num_swap_false < 720:
    print('TEST', i, '3*#swaps:', num_swap_true*3, num_swap_false*3)
        
time_used = round(time.time()-timeA,2)
print(time_used)

#for i in range(repeat):
#    num_swap, depth_overhead = execute_sabre(qc, coupling_map)
#    
#    if  depth_overhead == best_value: 
#        print(num_swap, depth_overhead)

grover_operator_6.qasm: qubit number=6, input depth=162, 
  ----- OrderedDict([('u1', 93), ('cx', 92), ('u', 27), ('u3', 2)])
The total CNOT overhead is 51.
TEST 0 3*#swaps: 45 57
The total CNOT overhead is 51.
TEST 1 3*#swaps: 45 63
The total CNOT overhead is 51.
TEST 2 3*#swaps: 45 57
The total CNOT overhead is 69.
TEST 3 3*#swaps: 69 60
The total CNOT overhead is 51.
TEST 4 3*#swaps: 45 57
The total CNOT overhead is 51.
TEST 5 3*#swaps: 45 63
The total CNOT overhead is 51.
TEST 6 3*#swaps: 45 57
The total CNOT overhead is 51.
TEST 7 3*#swaps: 45 57
The total CNOT overhead is 51.
TEST 8 3*#swaps: 45 63
The total CNOT overhead is 51.
TEST 9 3*#swaps: 45 63
0.66


In [ ]:
import json, os, time
import numpy as np
from ag import Sycamore54Q
AG = q20()
coupling_map = CouplingMap(AG.edges())

#path = '../bench/effectiveQM/'

path = '../bench/qiskit_circuit_benchmark/'
repeat = 10
obj = 'swap_overhead'
count = 0
Result = dict()
for filename in os.listdir(path):
    timeA = time.time()
    count += 1
    if filename[-4:] != 'qasm': continue
    #if '50.qasm' not in filename: continue
    #if '20QBT_gate_Tokyo_large_opt1_20' not in filename: continue
        
    """Create a QuantumCircuit from the QASM file"""
    with open(path+filename, 'r') as file:
        qasm_code = file.read()
    qc = QuantumCircuit.from_qasm_str(qasm_code)

    #qc = remove_1q_and_consecutive_2q_gates_in_circuit(circ)

    #dag = circuit_to_dag(newcirc)
    #print(filename, dag.count_ops(), dag.depth())

    print(f"Circ.{count}  {filename}: qubit number={qc.num_qubits}, {qc.count_ops()}")
    if qc.num_qubits > len(AG.nodes()): 
        continue
    #if qc.count_ops()['cx'] > 100: continue
    Result[filename] = [np.inf,np.inf]
    for i in range(repeat):
        num_swap_true, num_swap_false = execute_sabre(qc, coupling_map)
        print('TEST', i, '3*#swaps:', num_swap_true*3, num_swap_false*3)
        if num_swap_true < Result[filename][0]:
            Result[filename][0] = num_swap_true
        if num_swap_false < Result[filename][1]:
            Result[filename][1] = num_swap_false
    time_used = round(time.time()-timeA,2)
    print(Result[filename], time_used)